In [126]:
DEBUG = True
RANDOM_STATE = 2024

In [127]:
import matplotlib.pyplot as plt
import math
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# 1. Набор данных

In [128]:
path = './Titanic-Dataset-Processed.csv'
data = pd.read_csv(path, sep=',')

X = data.drop("Survived", axis=1)
y = data["Survived"]

X_train_val, X_test, y_train_val, y_test = train_test_split(X.to_numpy(), y.to_numpy(), test_size=0.25,
                                                            random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.3, random_state=RANDOM_STATE)

print(f'Размер тренировочной выборки: {X_train.shape}')
print(f'Размер валидационной выборки: {X_val.shape}')
print(f'Размер тестовой выборки: {X_test.shape}')

In [129]:
X.head()

# 2. Алгоритм

### 2.1. Дерево принятия решений

In [130]:
class MyDecisionTree:
    def __init__(self, max_depth=None, min_samples_split=2, min_samples_leaf=1, max_features=None, need_prune=True, random_state=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.need_prune = need_prune
        self.random_state = random_state
        self.n_features_ = None
        self.tree_ = None

    def fit(self, X, y, X_val=None, y_val=None):
        self.n_features_ = X.shape[1]
        if self.max_features is None or self.max_features > self.n_features_:
            self.max_features = self.n_features_

        if self.need_prune and (X_val is None or y_val is None):
            X, X_val, y, y_val = train_test_split(X, y, test_size=0.3, random_state=self.random_state)

        if self.random_state is not None:
            np.random.seed(self.random_state)

        self.tree_ = self._build_tree(X, y)

        if self.need_prune:
            self._prune_tree(self.tree_, X_val, y_val)

    def predict(self, X):
        return np.array([self._predict_single(x, self.tree_) for x in X])

    def score(self, X, y):
        y_pred = self.predict(X)
        return np.mean(y_pred == y)

    def get_height(self):
        def _compute_height(node):
            if node['type'] == 'leaf':
                return 1
            left_height = _compute_height(node['left'])
            right_height = _compute_height(node['right'])
            return 1 + max(left_height, right_height)

        if self.tree_ is None:
            return 0
        return _compute_height(self.tree_)

    def _build_tree(self, X, y, depth=0):
        # Условие остановки
        if len(y) < self.min_samples_split or depth == self.max_depth or len(set(y)) == 1:
            return {'type': 'leaf', 'class': self._majority_class(y)}

        # Найти лучшее разделение
        best_split = self._find_best_split(X, y)
        if not best_split:
            return {'type': 'leaf', 'class': self._majority_class(y)}

        left_idx, right_idx = best_split['left'], best_split['right']
        # Условие остановки, если одно из множеств слишком мало
        if len(left_idx) < self.min_samples_leaf or len(right_idx) < self.min_samples_leaf:
            return {'type': 'leaf', 'class': self._majority_class(y)}

        # Рекурсивно строим дерево
        left_subtree = self._build_tree(X[left_idx], y[left_idx], depth + 1)
        right_subtree = self._build_tree(X[right_idx], y[right_idx], depth + 1)

        return {
            'type': 'node',
            'feature': best_split['feature'],
            'threshold': best_split['threshold'],
            'left': left_subtree,
            'right': right_subtree
        }

    def _find_best_split(self, X, y, n_thresholds = 20):
        best_split = None
        best_criterion = -float('inf')

        # Выбираем случайное подмножество признаков
        feature_indices = np.random.choice(self.n_features_, self.max_features, replace=False)

        for feature in feature_indices:
            unique_values = np.unique(X[:, feature])
            thresholds = np.linspace(unique_values.min(), unique_values.max(), n_thresholds + 1)[1:-1]
            for threshold in thresholds:
                left_idx = np.where(X[:, feature] <= threshold)[0]
                right_idx = np.where(X[:, feature] > threshold)[0]

                if len(left_idx) == 0 or len(right_idx) == 0:
                    continue

                criterion = self._gini_gain(y[left_idx], y[right_idx])
                if criterion > best_criterion:
                    best_criterion = criterion
                    best_split = {
                        'feature': feature,
                        'threshold': threshold,
                        'left': left_idx,
                        'right': right_idx
                    }

        return best_split

    def _gini_gain(self, left_y, right_y):
        y = np.concatenate((left_y, right_y))
        return self._gini(y) - (
                (len(left_y) / len(y)) * self._gini(left_y) +
                (len(right_y) / len(y)) * self._gini(right_y)
        )

    def _gini(self, y):
        classes, counts = np.unique(y, return_counts=True)
        probs = counts / len(y)
        return 1 - np.sum(probs ** 2)

    def _majority_class(self, y):
        classes, counts = np.unique(y, return_counts=True)
        return classes[np.argmax(counts)]

    def _prune_tree(self, tree, X_val, y_val):
        if X_val.size == 0 or y_val.size == 0:
            return

        if tree['type'] == 'leaf':
            return

        left_mask = X_val[:, tree['feature']] <= tree['threshold']
        right_mask = ~left_mask

        # Рекурсивная подрезка левого и правого поддерева
        self._prune_tree(tree['left'], X_val[left_mask], y_val[left_mask])
        self._prune_tree(tree['right'], X_val[right_mask], y_val[right_mask])

        # Проверка качества после подрезки
        if tree['left']['type'] == 'leaf' and tree['right']['type'] == 'leaf':
            # Создать предсказания, если объединить узлы в один
            combined_class = self._majority_class(y_val)
            combined_predictions = np.full(y_val.shape, combined_class)

            # Проверяем качество до и после подрезки
            before_prune_acc = np.mean(self.predict(X_val) == y_val)
            after_prune_acc = np.mean(combined_predictions == y_val)

            # Если качество не ухудшилось, подрезаем
            if after_prune_acc >= before_prune_acc:
                tree['type'] = 'leaf'
                tree['class'] = combined_class
                tree.pop('left', None)
                tree.pop('right', None)
                tree.pop('feature', None)
                tree.pop('threshold', None)

    def _predict_single(self, x, tree):
        if tree['type'] == 'leaf':
            return tree['class']

        if x[tree['feature']] <= tree['threshold']:
            return self._predict_single(x, tree['left'])
        else:
            return self._predict_single(x, tree['right'])


### 2.2. Случайный лес

In [131]:
class MyRandomForest:
    def __init__(self, n_estimators=10, random_state=None):
        self.n_estimators = n_estimators
        self.random_state = random_state
        self.trees = []
        self.max_features = None

    def fit(self, X, y):
        if self.random_state is not None:
            np.random.seed(self.random_state)

        n_samples, n_features = X.shape
        self.max_features = int(np.sqrt(n_features))

        self.trees = []

        for _ in range(self.n_estimators):
            # Выбираем случайное подмножество с повторениями
            indices = np.random.choice(n_samples, n_samples, replace=True)
            X_subset = X[indices]
            y_subset = y[indices]

            # Выбираем случайные признаки
            feature_indices = np.random.choice(n_features, self.max_features, replace=False)

            # Обучаем MyDecisionTree
            tree = MyDecisionTree(need_prune=False)
            tree.fit(X_subset[:, feature_indices], y_subset)

            # Сохраняем дерево и выбранные признаки
            self.trees.append((tree, feature_indices))

    def predict(self, X):
        predictions = np.array([
            tree.predict(X[:, feature_indices]) for tree, feature_indices in self.trees
        ])
        majority_vote = stats.mode(predictions, axis=0).mode.flatten()
        return majority_vote

    def score(self, X, y):
        y_pred = self.predict(X)
        return np.mean(y_pred == y)


# 3. Задачи

### 3.1. Зависимость высоты дерева от числовых гиперпараметров (без ограничения высоты дерева) | MyDecisionTree + LibDecisionTree

In [132]:
# Переменные для хранения высот дерева
heights_split_lib = []
heights_leaf_lib = []
heights_features_lib = []

# Гиперпараметры для перебора
min_samples_split_values = range(2, 21)
min_samples_leaf_values = range(1, 21)
max_features_values = range(1, X_train.shape[1] + 1)

# Перебор min_samples_split
for value in tqdm(min_samples_split_values, desc="min_samples_split"):
    tree = DecisionTreeClassifier(min_samples_split=value, random_state=RANDOM_STATE)
    tree.fit(X_train, y_train)
    heights_split_lib.append(tree.tree_.max_depth)

# Перебор min_samples_leaf
for value in tqdm(min_samples_leaf_values, desc="min_samples_leaf"):
    tree = DecisionTreeClassifier(min_samples_leaf=value, random_state=RANDOM_STATE)
    tree.fit(X_train, y_train)
    heights_leaf_lib.append(tree.tree_.max_depth)

# Перебор max_features
for value in tqdm(max_features_values, desc="max_features"):
    tree = DecisionTreeClassifier(max_features=value, random_state=RANDOM_STATE)
    tree.fit(X_train, y_train)
    heights_features_lib.append(tree.tree_.max_depth)

# Построение графиков
fig, ax = plt.subplots(1, 3, figsize=(18, 6))

# График для min_samples_split
ax[0].plot(min_samples_split_values, heights_split_lib, label="Высота дерева", marker='o')
ax[0].set_title("min_samples_split")
ax[0].set_xlabel("min_samples_split")
ax[0].set_ylabel("Высота дерева")
ax[0].set_xticks(min_samples_split_values)
ax[0].set_yticks(np.arange(math.floor(ax[0].get_yticks()[0]), math.ceil(ax[0].get_yticks()[-1]), 1))
ax[0].set_xticklabels(ax[0].get_xticks().astype(int))
ax[0].set_yticklabels(ax[0].get_yticks().astype(int))
ax[0].grid()

# График для min_samples_leaf
ax[1].plot(min_samples_leaf_values, heights_leaf_lib, label="Высота дерева", marker='o')
ax[1].set_title("min_samples_leaf")
ax[1].set_xlabel("min_samples_leaf")
ax[1].set_xticks(min_samples_leaf_values)
ax[1].set_yticks(np.arange(math.floor(ax[1].get_yticks()[0]), math.ceil(ax[1].get_yticks()[-1]), 1))
ax[1].set_xticklabels(ax[1].get_xticks().astype(int))
ax[1].set_yticklabels(ax[1].get_yticks().astype(int))
ax[1].grid()

# График для max_features
ax[2].plot(max_features_values, heights_features_lib, label="Высота дерева", marker='o')
ax[2].set_title("max_features")
ax[2].set_xlabel("max_features")
ax[2].set_xticks(max_features_values)
ax[2].set_yticks(np.arange(math.floor(ax[2].get_yticks()[0]), math.ceil(ax[2].get_yticks()[-1]), 1))
ax[2].set_xticklabels(ax[2].get_xticks().astype(int))
ax[2].set_yticklabels(ax[2].get_yticks().astype(int))
ax[2].grid()

print("Library DecisionTreeClassifier")
plt.tight_layout()
plt.show()


In [133]:
# Переменные для хранения высот дерева
heights_split_my = []
heights_leaf_my = []
heights_features_my = []

# Гиперпараметры для перебора
min_samples_split_values = range(2, 21)
min_samples_leaf_values = range(1, 21)
max_features_values = range(1, X_train.shape[1] + 1)

# Перебор min_samples_split
for value in tqdm(min_samples_split_values, desc="min_samples_split"):
    tree = MyDecisionTree(min_samples_split=value, random_state=RANDOM_STATE)
    tree.fit(X_train, y_train)
    heights_split_my.append(tree.get_height())

# Перебор min_samples_leaf
for value in tqdm(min_samples_leaf_values, desc="min_samples_leaf"):
    tree = MyDecisionTree(min_samples_leaf=value, random_state=RANDOM_STATE)
    tree.fit(X_train, y_train)
    heights_leaf_my.append(tree.get_height())

# Перебор max_features
for value in tqdm(max_features_values, desc="max_features"):
    tree = MyDecisionTree(max_features=value, random_state=RANDOM_STATE)
    tree.fit(X_train, y_train)
    heights_features_my.append(tree.get_height())

# Построение графиков
fig, ax = plt.subplots(1, 3, figsize=(18, 6))

# График для min_samples_split
ax[0].plot(min_samples_split_values, heights_split_my, label="Высота дерева", marker='o')
ax[0].set_title("min_samples_split")
ax[0].set_xlabel("min_samples_split")
ax[0].set_ylabel("Высота дерева")
ax[0].set_xticks(min_samples_split_values)
ax[0].set_yticks(np.arange(math.floor(ax[0].get_yticks()[0]), math.ceil(ax[0].get_yticks()[-1]), 1))
ax[0].set_xticklabels(ax[0].get_xticks().astype(int))
ax[0].set_yticklabels(ax[0].get_yticks().astype(int))
ax[0].grid()

# График для min_samples_leaf
ax[1].plot(min_samples_leaf_values, heights_leaf_my, label="Высота дерева", marker='o')
ax[1].set_title("min_samples_leaf")
ax[1].set_xlabel("min_samples_leaf")
ax[1].set_xticks(min_samples_leaf_values)
ax[1].set_yticks(np.arange(math.floor(ax[1].get_yticks()[0]), math.ceil(ax[1].get_yticks()[-1]), 1))
ax[1].set_xticklabels(ax[1].get_xticks().astype(int))
ax[1].set_yticklabels(ax[1].get_yticks().astype(int))
ax[1].grid()

# График для max_features
ax[2].plot(max_features_values, heights_features_my, label="Высота дерева", marker='o')
ax[2].set_title("max_features")
ax[2].set_xlabel("max_features")
ax[2].set_xticks(max_features_values)
ax[2].set_yticks(np.arange(math.floor(ax[2].get_yticks()[0]), math.ceil(ax[2].get_yticks()[-1]), 1))
ax[2].set_xticklabels(ax[2].get_xticks().astype(int))
ax[2].set_yticklabels(ax[2].get_yticks().astype(int))
ax[2].grid()

print("MyDecisionTree")
plt.tight_layout()
plt.show()


### 3.2. Зависимость качества/ошибки от высоты дерева (train/test) MyDecisionTree + LibDecisionTree

In [134]:
train_scores_lib = []
test_scores_lib = []
train_scores_my = []
test_scores_my = []
tree_heights = []

previous_height = 0
depths = range(1, 21)
for depth in depths:
    # Библиотечная реализация DecisionTreeClassifier
    lib_tree = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE)
    lib_tree.fit(X_train, y_train)
    train_scores_lib.append(lib_tree.score(X_train, y_train))
    test_scores_lib.append(lib_tree.score(X_test, y_test))

    # Свой алгоритм MyDecisionTree
    my_tree = MyDecisionTree(max_depth=depth)
    my_tree.fit(X_train, y_train, X_val, y_val)
    train_scores_my.append(my_tree.score(X_train, y_train))
    test_scores_my.append(my_tree.score(X_test, y_test))

    # Высота дерева
    current_height = lib_tree.tree_.max_depth
    if current_height == previous_height:
        train_scores_lib.pop()
        test_scores_lib.pop()
        train_scores_my.pop()
        test_scores_my.pop()
        break
    previous_height = current_height
    tree_heights.append(current_height)

# Создаем графики
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# График для тренировочного множества
axes[0].plot(tree_heights, train_scores_lib, label="Library DecisionTree (Train)", marker='o')
axes[0].plot(tree_heights, train_scores_my, label="MyDecisionTree (Train)", marker='o')
axes[0].set_title("Точность на тренировочном множестве")
axes[0].set_xlabel("Высота дерева")
axes[0].set_ylabel("Точность")
axes[0].legend()
axes[0].grid()

# График для тестового множества
axes[1].plot(tree_heights, test_scores_lib, label="Library DecisionTree (Test)", marker='o')
axes[1].plot(tree_heights, test_scores_my, label="MyDecisionTree (Test)", marker='o')
axes[1].set_title("Точность на тестовом множестве")
axes[1].set_xlabel("Высота дерева")
axes[1].set_ylabel("Точность")
axes[1].legend()
axes[1].grid()

plt.tight_layout()
plt.show()


### 3.3. Зависимость качества/ошибки от числа деревьев (train/test) MyRandomForest + LibRandomForest + LibBoosting

In [135]:
train_scores_my_rf = []
test_scores_my_rf = []
train_scores_lib_rf = []
test_scores_lib_rf = []
train_scores_boost = []
test_scores_boost = []

n_trees = range(1, 501, 10)
for n in tqdm(n_trees):
    # Качество для вашей реализации случайного леса
    my_rf = MyRandomForest(n_estimators=n, random_state=RANDOM_STATE)
    my_rf.fit(X_train, y_train)
    train_scores_my_rf.append(my_rf.score(X_train, y_train))
    test_scores_my_rf.append(my_rf.score(X_test, y_test))

    # Качество для библиотечного случайного леса
    lib_rf = RandomForestClassifier(n_estimators=n, random_state=RANDOM_STATE)
    lib_rf.fit(X_train, y_train)
    train_scores_lib_rf.append(lib_rf.score(X_train, y_train))
    test_scores_lib_rf.append(lib_rf.score(X_test, y_test))

    # Качество для бустинга
    boost = GradientBoostingClassifier(n_estimators=n, random_state=RANDOM_STATE)
    boost.fit(X_train, y_train)
    train_scores_boost.append(boost.score(X_train, y_train))
    test_scores_boost.append(boost.score(X_test, y_test))

# Построение графиков
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# График для тренировочного множества
axes[0].plot(n_trees, train_scores_my_rf, label="MyRandomForest (Train)", marker='o')
axes[0].plot(n_trees, train_scores_lib_rf, label="Library RandomForest (Train)", marker='o')
axes[0].plot(n_trees, train_scores_boost, label="Boosting (Train)", marker='o')
axes[0].set_title("Качество на тренировочном множестве")
axes[0].set_xlabel("Число деревьев")
axes[0].set_ylabel("Точность")
axes[0].legend()
axes[0].grid()

# График для тестового множества
axes[1].plot(n_trees, test_scores_my_rf, label="MyRandomForest (Test)", marker='o')
axes[1].plot(n_trees, test_scores_lib_rf, label="Library RandomForest (Test)", marker='o')
axes[1].plot(n_trees, test_scores_boost, label="Boosting (Test)", marker='o')
axes[1].set_title("Качество на тестовом множестве")
axes[1].set_xlabel("Число деревьев")
axes[1].set_ylabel("Точность")
axes[1].legend()
axes[1].grid()

plt.tight_layout()
plt.show()
